In [3]:
from keras.applications.mobilenet import MobileNet
from keras.applications.mobilenet import preprocess_input
from keras.applications.mobilenet import decode_predictions
from PIL import Image
import numpy as np
import gradio as gr

# 建立 MobileNet 模型
model = MobileNet(weights="imagenet", include_top=True) 

def resize_image(img, new_w, new_h):
    # 轉換成 PIL 圖形
    img = Image.fromarray(img)    
    # 取得圖片的原始尺寸
    w, h = img.size
    # 計算長寬比例
    w_scale = new_w / w
    h_scale = new_h / h 
    scale = min(w_scale, h_scale)
    # 調整圖片尺寸
    resized = img.resize((int(w*scale), int(h*scale)), Image.NEAREST)    
    # 剪裁成正確的尺寸
    resized = resized.crop((0, 0, new_w, new_h))
    return resized

def predict(input):
    input_resized = resize_image(input, 224, 224)
    # 轉換圖片成為NumPy陣列
    img = np.array(input_resized)
    # 轉換成4D張量(1, 244, 244, 3)
    img = img.reshape((1, 224, 224, 3))
    # 資料預處理
    img = preprocess_input(img)    
    # 使用模型進行預測
    y_pred = model.predict(img, verbose=0)
    # 解碼預測結果
    label = decode_predictions(y_pred)
    top_prediction = label[0][0]  # 取得最可能的結果
    formatted_string = "%s (%.2f%%)" % (top_prediction[1], top_prediction[2]*100)
    return formatted_string
    
app = gr.Interface(fn=predict, 
                   inputs=gr.Image(), 
                   outputs="text")
app.launch()

Running on local URL:  http://127.0.0.1:7863

To create a public link, set `share=True` in `launch()`.
